# Entrenamiento real en GPU (Colab) — score_matching

Este notebook **no reimplementa nada**: clona el repo real de GitHub y
ejecuta `download_data.py`, `train.py` y `train_clf.py` tal cual estan en el
repositorio, para trazabilidad limpia (el checkpoint resultante queda ligado
a un commit concreto, no a una copia pegada en una celda).

**Antes de correr**: `Entorno de ejecucion -> Cambiar tipo de entorno de
ejecucion -> GPU`.

In [ ]:
import torch

print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("ADVERTENCIA: sin GPU. Entorno de ejecucion -> Cambiar tipo -> GPU.")

## Clonar (o actualizar) el repo real

In [ ]:
import os

REPO_URL = "https://github.com/anmedinas/score_matching.git"
REPO_DIR = "/content/score_matching"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo ya clonado, actualizando con git pull...")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
!git log --oneline -1

## Dependencias

Se instala `requirements.txt` **sin** `torch`/`torchvision`: Colab ya trae
una version con soporte CUDA preinstalada; reinstalar la version pineada del
repo (CPU por defecto desde PyPI) rompería la GPU.

In [ ]:
!grep -v -E '^torch' requirements.txt > /tmp/requirements_colab.txt
!pip install -q -r /tmp/requirements_colab.txt

## Datos

In [ ]:
!python download_data.py

## Hiperparametros (editables)

Ajusta y vuelve a correr esta celda antes de entrenar. `label_dropout_cfg`
queda en 0.1 (valor estandar en la literatura de CFG); el resto son puntos
de partida razonables para GPU, no valores finales.

In [ ]:
n_epochs_diffusion = 10  #@param {type:"integer"}
n_epochs_clf = 5  #@param {type:"integer"}
batch_size = 128  #@param {type:"integer"}
lr = 2e-4  #@param {type:"number"}
lr_clf = 1e-3  #@param {type:"number"}
label_dropout_cfg = 0.1  #@param {type:"number"}
seed = 0  #@param {type:"integer"}
n_sampling_steps = 1000  #@param {type:"integer"}
w_cfg = 3.0  #@param {type:"number"}

## Entrenamiento — red condicional pura (`modelo_cond.pt`, `label_dropout=0.0`)

In [ ]:
!python train.py \
  --epochs {n_epochs_diffusion} --batch-size {batch_size} --lr {lr} \
  --device cuda --seed {seed} --label-dropout 0.0 --out checkpoints

## Entrenamiento — red con CFG (`modelo_cfg.pt`, `label_dropout={label_dropout_cfg}`)

In [ ]:
!python train.py \
  --epochs {n_epochs_diffusion} --batch-size {batch_size} --lr {lr} \
  --device cuda --seed {seed} --label-dropout {label_dropout_cfg} --out checkpoints

## Entrenamiento — clasificador auxiliar (`clasificador.pt`)

In [ ]:
!python train_clf.py \
  --epochs {n_epochs_clf} --batch-size {batch_size} --lr {lr_clf} \
  --device cuda --seed {seed} --out checkpoints

## Curvas de perdida (generadas por los scripts, solo se muestran aqui)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

loss_files = ["modelo_cond_loss.png", "modelo_cfg_loss.png", "clasificador_loss.png"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, name in zip(axes, loss_files):
    ax.imshow(Image.open(f"figures/loss/{name}"))
    ax.axis("off")
    ax.set_title(name)
plt.tight_layout()
plt.show()

## Muestreo (opcional)

Requiere que `sample.py` este pusheado al repo remoto. Si todavia no lo
esta, esta celda lo detecta y avisa en vez de fallar; sube `sample.py`,
vuelve a correr la celda de `git pull` de mas arriba, y reintenta.

In [ ]:
import os

if not os.path.exists("sample.py"):
    print("sample.py todavia no esta en el repo remoto (no fue pusheado).")
    print("Subelo, vuelve a correr la celda de clonado/pull, y reintenta esta celda.")
else:
    !python sample.py --checkpoint checkpoints/modelo_cond.pt \
      --n-steps {n_sampling_steps} --w 1.0 --n-samples 30 --device cuda \
      --seed {seed} --out figures/samples
    !python sample.py --checkpoint checkpoints/modelo_cfg.pt \
      --n-steps {n_sampling_steps} --w 1.0 --n-samples 30 --device cuda \
      --seed {seed} --out figures/samples
    !python sample.py --checkpoint checkpoints/modelo_cfg.pt \
      --n-steps {n_sampling_steps} --w {w_cfg} --n-samples 30 --device cuda \
      --seed {seed} --out figures/samples

    import matplotlib.pyplot as plt
    from PIL import Image

    sample_files = ["modelo_cond_w1.png", "modelo_cfg_w1.png", f"modelo_cfg_w{w_cfg:g}.png"]
    fig, axes = plt.subplots(1, 3, figsize=(10, 14))
    for ax, name in zip(axes, sample_files):
        ax.imshow(Image.open(f"figures/samples/{name}"))
        ax.axis("off")
        ax.set_title(name)
    plt.tight_layout()
    plt.show()

## Descargar resultados

Empaqueta `checkpoints/` y `figures/` en un .zip para bajarlos al equipo
local (no se hace `git push` desde aca; el repo local sigue siendo la fuente
de verdad que tu decides cuando subir).

In [ ]:
from google.colab import files

!zip -r -q /content/resultados_colab.zip checkpoints figures
files.download("/content/resultados_colab.zip")